# Phase 2: Environment Setup

**Goal:** confirm GR00T N1.7 (action/System1) and Cosmos-Reason2-2B (reasoning proxy/System2) both load and run for inference on Colab Pro, pull a small subset (10-20 trajectories) of `nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1` (real Unitree G1 teleop data), and verify the data format end-to-end. Nothing here trains or fine-tunes anything — inference only.

**Dataset note (2026-07-17):** originally planned around AgiBot World, but `EmbodimentTag.AGIBOT_GENIE1` does not exist in the current GR00T codebase — AgiBot has no zero-shot pretrain tag, so using it would have required fine-tuning, which is off the table under our compute constraints. Pivoted to `nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1`, NVIDIA's own published Unitree G1 teleop dataset, which matches GR00T's genuine `REAL_G1` zero-shot pretrain tag by construction. Bonus: this dataset's state/action space includes full leg and waist joints (a real bipedal humanoid), which AgiBot's wheeled base never could have given us — see the note in Section 6/7 about deriving sub-goal boundaries from hand-joint kinematics rather than AgiBot-style pre-labeled segments.

**Runtime:** Runtime > Change runtime type > GPU > A100 (Colab Pro/Pro+), and select the **High-RAM (80GB)** option if offered. GR00T N1.7 needs ~16GB and Cosmos-Reason2-2B needs a NVIDIA-stated **minimum of 24GB** (corrected 2026-07-17 — Phase 1's ~4-6GB estimate was a naive param-count guess, not NVIDIA's own figure) — with both loaded simultaneously for the smoke test in Section 7, that's ~40GB, right at or over a standard 40GB A100's limit.

**Checkpointing:** everything that takes real time (downloads, model loads, verification results) writes a status marker to Google Drive so a disconnected session can resume without redoing completed steps. Re-run all cells top to bottom after a reconnect — completed steps will just print "already done" and skip.

**If any cell errors: stop, copy the full error output, and paste it back.** Do not try to fix it yourself — hand it back so the underlying assumption can be corrected against the real error rather than guessed again.

## 0. GPU check, Drive mount, checkpoint scaffolding

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time

PROJECT_DIR = '/content/drive/MyDrive/humanoid-vla-eval'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'g1_teleop_subset')
STATUS_PATH = os.path.join(CKPT_DIR, 'phase2_status.json')

os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)

def load_status():
    if os.path.exists(STATUS_PATH):
        with open(STATUS_PATH) as f:
            return json.load(f)
    return {}

def save_status(key, value):
    status = load_status()
    status[key] = value
    status['_last_updated'] = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(STATUS_PATH, 'w') as f:
        json.dump(status, f, indent=2)
    return status

def clear_status(key=None):
    """Manual escape hatch: clear_status() wipes everything, clear_status('some_key')
    wipes just that key. Use this if a checkpoint flag is ever suspected stale —
    the 'done' cells below now self-verify instead of blindly trusting the flag, so
    this shouldn't be needed in normal operation, but it's here for a manual override."""
    status = {} if key is None else {k: v for k, v in load_status().items() if k != key}
    with open(STATUS_PATH, 'w') as f:
        json.dump(status, f, indent=2)
    print('Cleared:', key or 'ALL status')

def is_done(key):
    return load_status().get(key) == 'done'

print('Current status:', load_status())

## 1. Install dependencies

In [ ]:
import subprocess, platform, importlib, sys

def run(cmd):
    """Run a shell command and raise loudly on failure — a bare `!cmd` in Colab does
    NOT raise on nonzero exit, which is why an earlier version of this cell silently
    marked a failed install as done."""
    print(f'$ {cmd}')
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f'Command failed (exit {result.returncode}): {cmd}')

GR00T_REPO_PATH = '/content/Isaac-GR00T'
RESTART_MARKER = '/content/.numpy_clean_reinstall_done'

def clean_reinstall_numpy():
    """Force a genuinely clean numpy install ON DISK.

    numpy 2.0 renamed its internal `numpy/core/` directory to `numpy/_core/`. Colab ships
    numpy 2.x by default; when uv resolves gr00t's pin (numpy==1.26.4, pre-2.0, which uses
    `numpy/core/`) it upgrades/downgrades numpy IN PLACE inside the same site-packages
    directory. That in-place swap does not reliably remove the old version's directory
    layout, so both `core/` (new, from 1.26.4) and `_core/` (orphaned, from the original
    2.x) can end up coexisting on disk. We find numpy's actual install directory and
    delete it outright before reinstalling from scratch, so no orphaned files survive.

    NOTE: this fixes the FILES on disk only. It deliberately does NOT try to make numpy
    re-importable within the current process — see gr00t_importable()'s docstring for why
    that's not something Python can do safely for a compiled C-extension module.
    """
    result = subprocess.run(
        [sys.executable, '-c', 'import numpy, os; print(os.path.dirname(numpy.__file__))'],
        capture_output=True, text=True,
    )
    if result.returncode == 0 and result.stdout.strip():
        numpy_dir = result.stdout.strip()
        print(f'Found existing numpy install at: {numpy_dir} — removing it entirely.')
        run(f'rm -rf "{numpy_dir}"')
    else:
        print('No existing numpy import found (or already removed) — nothing to clean.')
    run(f'"{sys.executable}" -m pip uninstall -y numpy')
    run(f'uv pip install --python "{sys.executable}" --reinstall --no-cache numpy==1.26.4')
    with open(RESTART_MARKER, 'w') as f:
        f.write('numpy was reinstalled cleanly on disk; the CURRENT process may still have a '
                'stale copy loaded and needs a restart to pick it up.')

def gr00t_deep_import():
    """Try the real import chain cell 9 needs, in THIS process, as-is.

    Deliberately does NOT delete numpy from sys.modules and re-import it as a "fix" —
    that was tried and failed. numpy is a compiled C extension; once its shared library
    has been initialized once in a process, Python re-importing it after an
    sys.modules deletion does NOT safely reset the underlying C state (numpy itself
    warns "The NumPy module was reloaded... this can result in small but subtle issues
    and is discouraged" when this happens). If numpy was ever imported in this process
    before the on-disk version was corrected, no amount of further in-process
    reinstalling can fix it — only a genuine kernel restart (a fresh process) can.

    Returns (ok: bool, needs_restart: bool).
    """
    if os.path.isdir(GR00T_REPO_PATH) and GR00T_REPO_PATH not in sys.path:
        sys.path.insert(0, GR00T_REPO_PATH)
    importlib.invalidate_caches()
    try:
        import gr00t  # noqa: F401
        import gr00t.policy  # noqa: F401
        import gr00t.data.embodiment_tags  # noqa: F401
        import pandas  # noqa: F401
        import pyarrow  # noqa: F401 — pandas.read_parquet's engine, used in Section 6
        import cv2  # noqa: F401
        return True, False
    except ImportError:
        return False, False
    except ValueError as e:
        # numpy's ABI mismatch ("dtype size changed... binary incompatibility") raises
        # ValueError, not ImportError — this is the "needs a real restart" signal.
        print(f'numpy ABI mismatch in this process: {e}')
        return False, True

def print_diagnostics():
    print('\n--- DIAGNOSTICS ---')
    print('Kernel sys.executable:', sys.executable)
    print()
    subprocess.run(f'"{sys.executable}" -m pip show gr00t numpy pyarrow', shell=True)
    print()
    print('Fresh-subprocess import test (same interpreter as kernel):')
    result = subprocess.run([sys.executable, '-c', 'import gr00t.policy, pandas, pyarrow, cv2; import numpy; print("OK:", gr00t.__file__, numpy.__version__)'], capture_output=True, text=True)
    print('returncode:', result.returncode)
    print('stdout:', result.stdout)
    print('stderr:', result.stderr)

if os.path.exists(RESTART_MARKER):
    # A previous run of this cell already cleaned numpy on disk and asked for a restart.
    # If we're executing at all, either that restart happened (good — verify normally
    # below) or it didn't (the check below will catch that too).
    os.remove(RESTART_MARKER)

ok, needs_restart = gr00t_deep_import()

if ok:
    print('gr00t already importable — skipping install.')
    save_status('deps_installed', 'done')
elif needs_restart:
    print('numpy is already loaded in this running kernel in a way that cannot be fixed '
          'without a restart. Cleaning the on-disk install now so a restart picks up a '
          'consistent state immediately...')
    clean_reinstall_numpy()
    raise RuntimeError(
        'ACTION REQUIRED: Runtime > Restart session (NOT "Restart and run all" — just '
        'restart), then re-run this cell. The on-disk files are now correct, so the '
        'restarted process should pass this check immediately with no reinstall needed.'
    )
else:
    if is_done('deps_installed'):
        print("Checkpoint says deps_installed=done but the deep import fails — "
              "stale checkpoint from an earlier failed install. Reinstalling for real.")

    py_ver = platform.python_version()
    print(f'Python version: {py_ver}')
    print(f'Kernel sys.executable: {sys.executable}')
    if not py_ver.startswith('3.12'):
        print('WARNING: Isaac-GR00T requires Python 3.12.x (pyproject.toml: >=3.12,<3.13). '
              'If the install below fails with a python-version error, stop and paste it back.')

    if not os.path.exists(GR00T_REPO_PATH):
        run(f'git clone https://github.com/NVIDIA/Isaac-GR00T.git {GR00T_REPO_PATH}')

    # NVIDIA's pyproject.toml pins flash-attn to a prebuilt wheel URL via
    # [tool.uv.sources], which only `uv` resolves. Plain `pip install -e .` tries to
    # compile flash-attn from source instead and fails on Colab. We point uv at the
    # exact interpreter this kernel is running (sys.executable) rather than relying
    # on `--system` to guess the right one.
    run('pip install -q -U uv')
    run(f'uv pip install --python "{sys.executable}" -e {GR00T_REPO_PATH}')
    # accelerate (Cosmos-Reason2 device_map='auto') and pyarrow (pandas.read_parquet's
    # engine, Section 6) aren't in gr00t's own pin list — pyarrow is very likely already
    # present transitively via the `datasets` package, but pinning it explicitly removes
    # the ambiguity rather than assuming.
    run(f'uv pip install --python "{sys.executable}" accelerate pyarrow')

    ok, needs_restart = gr00t_deep_import()
    if needs_restart:
        print('numpy ABI mismatch surfaced during THIS install (something in this fresh-ish '
              'process already touched numpy before the pinned version landed). Cleaning the '
              'on-disk install so a restart picks up a consistent state...')
        clean_reinstall_numpy()
        raise RuntimeError(
            'ACTION REQUIRED: Runtime > Restart session, then re-run this cell. The on-disk '
            'files are now correct — the restarted process should pass immediately.'
        )
    elif not ok:
        print_diagnostics()
        raise RuntimeError(
            'Install commands reported success but the deep import still fails for a reason '
            'other than the known numpy ABI issue. Copy the full output of this cell '
            '(including the DIAGNOSTICS section) back to Claude.'
        )

    save_status('deps_installed', 'done')
    print('Dependencies installed and verified importable.')

## 2. Hugging Face auth

Only Cosmos-Reason2-2B is gated. `nvidia/GR00T-N1.7-3B` and
`nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1` (our dataset — see Section 5) are both
public, no license click-through needed.

1. Accept the license/terms while logged into your HF account:
   - https://huggingface.co/nvidia/Cosmos-Reason2-2B
2. Generate a read-access token at https://huggingface.co/settings/tokens
3. Paste it below when prompted (input is hidden).

In [ ]:
from huggingface_hub import login
login()

## 3. Load GR00T N1.7 (System 1 / action model)

Uses `EmbodimentTag.REAL_G1` (real-world Unitree G1), a genuine zero-shot pretrain tag
baked into the base checkpoint — confirmed against NVIDIA's `getting_started/policy.md`.
Our dataset (Section 5) is `nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1`, NVIDIA's own
published Unitree G1 teleop data, so this should be a clean match by construction rather
than an adapter guess (AgiBot's wheeled-base embodiment has no such tag and was dropped —
see project memory / conversation history for why).

In [ ]:
import torch

GR00T_CHECKPOINT = 'nvidia/GR00T-N1.7-3B'

gr00t_policy = None
gr00t_error = None
try:
    from gr00t.policy import Gr00tPolicy
    from gr00t.data.embodiment_tags import EmbodimentTag

    print('Available embodiment tags:', [t.name for t in EmbodimentTag])

    gr00t_policy = Gr00tPolicy(
        model_path=GR00T_CHECKPOINT,
        embodiment_tag=EmbodimentTag.REAL_G1,
        device='cuda' if torch.cuda.is_available() else 'cpu',
    )
    print('GR00T N1.7 loaded successfully with EmbodimentTag.REAL_G1.')
    save_status('gr00t_loaded', 'done')

    print('\n--- Modality config for REAL_G1 ---')
    gr00t_modality_configs = gr00t_policy.get_modality_config()
    for modality, cfg in gr00t_modality_configs.items():
        print(f'{modality}: keys={cfg.modality_keys}, horizon={len(cfg.delta_indices)}')
except Exception as e:
    gr00t_error = e
    print('GR00T load FAILED — copy this full traceback back to Claude:')
    raise

## 4. Load Cosmos-Reason2-2B (System 2 reasoning proxy)

This is run as a **standalone natural-language chain-of-thought reasoner**, not as part of GR00T's internal pipeline. It is a proxy for GR00T's true (inaccessible, latent-only) internal reasoning representation — same backbone lineage, but not literally the same weights or the same computation GR00T performs internally. This must stay labeled as a proxy everywhere downstream.

**VRAM correction (2026-07-17):** Phase 1 estimated ~4-6GB for this model from raw
parameter count. NVIDIA's own repo states a **minimum of 24GB** for Cosmos-Reason2-2B —
vision-language models have much larger activation/KV-cache footprints than parameter
count alone suggests, especially for multi-frame/video input. Combined with GR00T's
~16GB (already loaded in the previous cell, staying resident), that's ~40GB if both are
loaded at once — right at or over the edge of a standard 40GB A100. **If your Colab Pro
runtime offers a "High-RAM" A100 (80GB) option, select it now** (Runtime > Change runtime
type) rather than discovering an OOM mid-run. If only 40GB is available, we may need to
free GR00T from GPU memory before loading Cosmos-Reason2, or vice versa, in later phases.

Uses NVIDIA's exact reference loading code (`Qwen3VLForConditionalGeneration`, not the
generic `AutoModelForCausalLM` — Cosmos-Reason2 is a vision-language model on the
Qwen3-VL architecture, and the generic causal-LM class doesn't recognize its config
class, as the previous run's traceback confirmed).

In [ ]:
COSMOS_REASON2_CHECKPOINT = 'nvidia/Cosmos-Reason2-2B'

import transformers

cosmos_model = None
cosmos_processor = None
PIXELS_PER_TOKEN = 32 ** 2

try:
    cosmos_model = transformers.Qwen3VLForConditionalGeneration.from_pretrained(
        COSMOS_REASON2_CHECKPOINT,
        dtype=torch.float16,
        device_map='auto',
        attn_implementation='sdpa',
    )
    cosmos_processor = transformers.Qwen3VLProcessor.from_pretrained(COSMOS_REASON2_CHECKPOINT)

    # Bound vision tokens to control memory usage (NVIDIA's reference values)
    min_vision_tokens, max_vision_tokens = 256, 8192
    cosmos_processor.image_processor.size = {
        'shortest_edge': min_vision_tokens * PIXELS_PER_TOKEN,
        'longest_edge': max_vision_tokens * PIXELS_PER_TOKEN,
    }
    cosmos_processor.video_processor.size = {
        'shortest_edge': min_vision_tokens * PIXELS_PER_TOKEN,
        'longest_edge': max_vision_tokens * PIXELS_PER_TOKEN,
    }

    print('Cosmos-Reason2-2B loaded successfully.')
    save_status('cosmos_loaded', 'done')
except Exception as e:
    print('Cosmos-Reason2 load FAILED — copy this full traceback back to Claude:')
    raise

## 5. Pull a small subset of the G1 teleop dataset

`nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1`: 4 tasks (pick apple/pear/grapes/starfruit
and place on plate), ~311 episodes each, LeRobot v2.1 format. Public, not gated. We pull
one task folder's `meta/` (small) plus a handful of individual episode files (parquet +
mp4), not the full ~311-episode chunk — sparse enough to be a quick download and land
well under the 10-20 trajectory target.

Verified schema (fetched directly from the dataset's `meta/*.json` on 2026-07-17):
- `state`/`action`: 43-dim float64, split by `meta/modality.json` into
  `left_leg`(6) `right_leg`(6) `waist`(3) `left_arm`(7) `left_hand`(7) `right_arm`(7)
  `right_hand`(7) — this is a genuine whole-body space (legs + waist included), even
  though the demonstrated task is upper-body manipulation.
- `video`: `observation.images.ego_view`, 480x640x3, 20fps.
- `meta/tasks.jsonl`: one flat instruction per task (e.g. "Pick up the red apple and
  place it on the plate") — no AgiBot-style multi-level sub-goal segments. We'll derive
  sub-goal boundaries from hand-joint kinematic transitions in Phase 4 instead.

In [ ]:
from huggingface_hub import HfApi

G1_DATASET_REPO = 'nvidia/PhysicalAI-Robotics-GR00T-Teleop-G1'
TASK_FOLDER = 'g1-pick-apple'
N_EPISODES = 15  # keeps us well under the 20-trajectory Phase 2 target

api = HfApi()
files = api.list_repo_files(G1_DATASET_REPO, repo_type='dataset')
task_folders = sorted({f.split('/')[0] for f in files if f.count('/') > 0 and not f.startswith('.')})
print('Task folders found:', task_folders)
assert TASK_FOLDER in task_folders, f'{TASK_FOLDER} not found among {task_folders} — pick one of these instead.'

In [ ]:
from huggingface_hub import snapshot_download
import json as _json

if not is_done('g1_meta_downloaded'):
    snapshot_download(
        repo_id=G1_DATASET_REPO,
        repo_type='dataset',
        allow_patterns=[f'{TASK_FOLDER}/meta/*'],
        local_dir=DATA_DIR,
    )
    save_status('g1_meta_downloaded', 'done')
else:
    print('Task meta already downloaded — skipping.')

with open(os.path.join(DATA_DIR, TASK_FOLDER, 'meta', 'info.json')) as f:
    info = _json.load(f)

discarded = set(info.get('discarded_episode_indices', []))
total_episodes = info['total_episodes']
print(f'{total_episodes} total episodes, {len(discarded)} discarded: {sorted(discarded)}')

# Pick the first N_EPISODES valid (non-discarded) episode indices
selected_episodes = []
i = 0
while len(selected_episodes) < N_EPISODES and i < total_episodes:
    if i not in discarded:
        selected_episodes.append(i)
    i += 1
print(f'Selected {len(selected_episodes)} episodes:', selected_episodes)
save_status('g1_selected_episodes', selected_episodes)

if not is_done('g1_episodes_downloaded'):
    allow_patterns = []
    for ep in selected_episodes:
        allow_patterns.append(f'{TASK_FOLDER}/data/chunk-000/episode_{ep:06d}.parquet')
        allow_patterns.append(f'{TASK_FOLDER}/videos/chunk-000/observation.images.ego_view/episode_{ep:06d}.mp4')
    snapshot_download(
        repo_id=G1_DATASET_REPO,
        repo_type='dataset',
        allow_patterns=allow_patterns,
        local_dir=DATA_DIR,
    )
    save_status('g1_episodes_downloaded', 'done')
else:
    print('Episode subset already downloaded — skipping.')

!du -sh {DATA_DIR}

## 6. Parse one trajectory's format

Confirm we can read: (a) the language instruction, (b) at least one visual frame,
(c) the state/action array — before building anything automated on top. We print the
actual parquet column structure before assuming anything about it, since LeRobot-format
column layout details (single array column vs. flattened per-dim columns) can vary by
release and are worth confirming against real data rather than guessing again.

In [ ]:
import pandas as pd
import numpy as np
import cv2

TASK_DIR = os.path.join(DATA_DIR, TASK_FOLDER)

with open(os.path.join(TASK_DIR, 'meta', 'modality.json')) as f:
    modality_spec = _json.load(f)

episodes_meta = {}
with open(os.path.join(TASK_DIR, 'meta', 'episodes.jsonl')) as f:
    for line in f:
        rec = _json.loads(line)
        episodes_meta[rec['episode_index']] = rec

EP = selected_episodes[0]
print(f'--- Episode {EP} ---')
print('Language task(s):', episodes_meta[EP]['tasks'])
print('Length (frames):', episodes_meta[EP]['length'])

parquet_path = os.path.join(TASK_DIR, 'data', 'chunk-000', f'episode_{EP:06d}.parquet')
df = pd.read_parquet(parquet_path)
print('\n--- Parquet structure ---')
print('Columns:', df.columns.tolist())
print('Shape:', df.shape)
print('Dtypes:\n', df.dtypes)
print('\nFirst row (raw):')
for col in df.columns:
    val = df.iloc[0][col]
    preview = val if not hasattr(val, '__len__') or isinstance(val, str) else f'{type(val).__name__} len={len(val)}'
    print(f'  {col}: {preview}')

video_path = os.path.join(TASK_DIR, 'videos', 'chunk-000', 'observation.images.ego_view', f'episode_{EP:06d}.mp4')
cap = cv2.VideoCapture(video_path)
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
ret, frame_bgr = cap.read()
cap.release()
print(f'\n--- Video ---')
print(f'{video_path}')
print(f'frame_count={n_frames}, fps={fps}, first_frame_read_ok={ret}, shape={frame_bgr.shape if ret else None}')

In [ ]:
def split_by_modality(vec, spec):
    """Split a flat 43-dim state/action vector into named substreams per modality.json."""
    vec = np.asarray(vec, dtype=np.float32)
    return {name: vec[bounds['start']:bounds['end']] for name, bounds in spec.items()}

try:
    state_vec = df.iloc[0]['observation.state']
    action_vec = df.iloc[0]['action']
    state_split = split_by_modality(state_vec, modality_spec['state'])
    action_split = split_by_modality(action_vec, modality_spec['action'])
    print('State split OK. Keys and dims:', {k: v.shape for k, v in state_split.items()})
    print('Action split OK. Keys and dims:', {k: v.shape for k, v in action_split.items()})
    print('\nleft_leg + right_leg + waist state (whole-body lower-body pose):')
    print(np.concatenate([state_split['left_leg'], state_split['right_leg'], state_split['waist']]))
except Exception as e:
    print('Modality split FAILED — this means the parquet column layout printed above does not '
          'match the assumed single-array-column format. Copy the full "Parquet structure" output '
          'from the previous cell back to Claude so the parsing logic can be corrected against the '
          'real layout instead of another guess.')
    raise

## 7. End-to-end smoke test (one episode, both models)

Builds one observation from the parsed episode and runs it through both GR00T
(action/System1) and Cosmos-Reason2 (reasoning proxy/System2), using GR00T's own
reported modality config to determine the exact expected keys and temporal horizon
rather than assuming they match the dataset's raw column names.

In [ ]:
task_instruction = episodes_meta[EP]['tasks'][0]
print('Task instruction:', task_instruction)

video_keys = gr00t_modality_configs['video'].modality_keys
state_keys = gr00t_modality_configs['state'].modality_keys
T_video = len(gr00t_modality_configs['video'].delta_indices)
T_state = len(gr00t_modality_configs['state'].delta_indices)
print(f'GR00T expects video_keys={video_keys} (T={T_video}), state_keys={state_keys} (T={T_state})')

our_state_names = set(modality_spec['state'].keys())
missing = set(state_keys) - our_state_names
if missing:
    raise ValueError(
        f'GR00T expects state keys {missing} that are not in this dataset\'s modality.json '
        f'split ({our_state_names}). Copy this output back to Claude — building an incomplete '
        f'observation dict and continuing would just produce a confusing downstream error '
        f'inside GR00T\'s own validation instead of a clear one here.'
    )

# --- video: read T_video frames, convert BGR->RGB ---
cap = cv2.VideoCapture(video_path)
frames = []
for _ in range(T_video):
    ret, f = cap.read()
    if not ret:
        break
    frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
cap.release()
while 0 < len(frames) < T_video:
    frames.append(frames[-1])  # pad by repeating last frame if episode shorter than horizon
video_arr = np.stack(frames, axis=0)[np.newaxis, ...]  # (1, T_video, H, W, 3) uint8
assert len(video_keys) == 1, f'Expected exactly one video key, got {video_keys}'
observation_video = {video_keys[0]: video_arr}

# --- state: T_state timesteps, split per modality.json, keyed by GR00T's own state_keys ---
assert T_state <= episodes_meta[EP]['length'], (
    f'Episode has only {episodes_meta[EP]["length"]} frames, fewer than the {T_state} '
    f'GR00T needs for its state horizon — pick a longer episode.'
)
state_rows = [np.asarray(df.iloc[t]['observation.state'], dtype=np.float32) for t in range(T_state)]
state_split_rows = [split_by_modality(row, modality_spec['state']) for row in state_rows]
observation_state = {
    k: np.stack([row[k] for row in state_split_rows], axis=0)[np.newaxis, ...]  # (1, T_state, D)
    for k in state_keys
}

# --- language ---
observation_language = {'task': [[task_instruction]]}

observation = {'video': observation_video, 'state': observation_state, 'language': observation_language}
print('\nBuilt observation shapes:')
print('  video:', {k: v.shape for k, v in observation_video.items()})
print('  state:', {k: v.shape for k, v in observation_state.items()})
print('  language:', observation_language)

print('\n--- Running GR00T get_action() ---')
gr00t_action, gr00t_info = gr00t_policy.get_action(observation)
for k, v in gr00t_action.items():
    print(f'  action[{k}]: shape={v.shape}, first step={v[0, 0, :]}')

print('\n--- Running Cosmos-Reason2 (reasoning proxy) on the same first frame ---')
# Uses NVIDIA's chat-template calling convention (see scripts/inference_sample.py in
# nvidia-cosmos/cosmos-reason2), not a plain processor(text=..., images=...) call —
# Qwen3-VL-family models expect a structured conversation. Image is written to a temp
# file and passed by path, matching NVIDIA's own example (which passes video by path
# rather than a decoded array) rather than guessing whether an in-memory PIL object is
# accepted directly.
from PIL import Image
import tempfile

first_frame_rgb = frames[0]  # RGB numpy array, HxWx3
tmp_frame_path = os.path.join(tempfile.gettempdir(), 'cosmos_smoke_test_frame.png')
Image.fromarray(first_frame_rgb).save(tmp_frame_path)

reasoning_prompt = (
    f'Task: "{task_instruction}". Describe the sub-goals needed to complete this task, '
    f'in order, as a short numbered list.'
)
conversation = [
    {'role': 'system', 'content': [{'type': 'text', 'text': 'You are a helpful assistant.'}]},
    {'role': 'user', 'content': [
        {'type': 'image', 'image': tmp_frame_path},
        {'type': 'text', 'text': reasoning_prompt},
    ]},
]
cosmos_inputs = cosmos_processor.apply_chat_template(
    conversation, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt',
).to(cosmos_model.device)

with torch.no_grad():
    cosmos_out_ids = cosmos_model.generate(**cosmos_inputs, max_new_tokens=200)
cosmos_out_trimmed = cosmos_out_ids[:, cosmos_inputs['input_ids'].shape[1]:]
cosmos_reasoning_text = cosmos_processor.batch_decode(
    cosmos_out_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False,
)[0]
print(cosmos_reasoning_text)

# --- checkpoint the raw smoke-test outputs for Phase 3 ---
smoke_test_result = {
    'task_folder': TASK_FOLDER,
    'episode_index': EP,
    'task_instruction': task_instruction,
    'gr00t_action_shapes': {k: list(v.shape) for k, v in gr00t_action.items()},
    'gr00t_first_action_step': {k: v[0, 0, :].tolist() for k, v in gr00t_action.items()},
    'cosmos_reasoning_proxy_text': cosmos_reasoning_text,
}
with open(os.path.join(CKPT_DIR, 'phase2_smoke_test.json'), 'w') as f:
    json.dump(smoke_test_result, f, indent=2)
save_status('smoke_test', 'done')
print('\nSaved smoke test result to checkpoints/phase2_smoke_test.json')

## 8. Status summary

In [ ]:
print(json.dumps(load_status(), indent=2))